# Batch GLI: interactive folder → aggregate CSV

1. **Pick a root folder** (browse or type path) — all `.xlsx` / `.xls` files are found (optionally in subfolders).
2. Compute **GLI** predicted, **LLN**, ULN, z-score, % predicted per parameter.
3. **Classify** each row: Normal / Obstruction / Restriction / Mixed (FEV₁/FVC + TLC vs LLN).
4. Save two CSVs: **full** (all original columns + GLI) and **summary** (test date, age, key measures, LLN, classification only).

Run the **setup** cell, then the **folder picker** cell. Optional summary cells below.

In [2]:
from datetime import datetime
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Project root (parent of notebooks/)
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "backend").exists():
    PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.app.services.batch_service import (
    build_compact_summary,
    discover_excel_files,
    process_excel_paths,
    read_pft_excel,
    save_batch_csv_outputs,
)
from backend.app.services.fields import suggest_mapping
from backend.app.services.gli_service import gli_service

# Warm-load GLI spline tables (requires data/reference/*.xlsx)
_ = gli_service.tlc_splines
_ = gli_service.spiro_splines

MODULES = ["spirometry", "lung_volumes"]
AUTO_MAPPING_PER_FILE = True
DEFAULT_OUTPUT_DIR = PROJECT_ROOT / "data/output"

# Edit these if you prefer typing a path instead of the folder picker
BATCH_ROOT = str(PROJECT_ROOT / "data/samples")
BATCH_RECURSIVE = True
BATCH_CSV_NAME = "aggregated_PFT_GLI.csv"

combined_df: pd.DataFrame | None = None

print("GLI reference tables loaded.")
print("Reference data:", PROJECT_ROOT / "data/reference")

GLI reference tables loaded.
Reference data: /Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/data/reference


## Select folder → run → CSV

**Run the cell below.** On macOS, a **Finder** folder picker opens (no Jupyter widgets).

- Cancel the picker → uses `BATCH_ROOT` from the setup cell.
- Or edit `BATCH_ROOT` there and run `run_gli_batch()` in a new cell.

In [3]:
def pick_folder(initial: str | None = None) -> str | None:
    """Folder dialog without ipywidgets. macOS: Finder picker; else tkinter or typed path."""
    import subprocess
    import sys

    start = initial or BATCH_ROOT or str(PROJECT_ROOT)

    if sys.platform == "darwin":
        script = (
            'POSIX path of (choose folder with prompt '
            '"Select root folder containing PFT Excel files")'
        )
        proc = subprocess.run(
            ["osascript", "-e", script],
            capture_output=True,
            text=True,
        )
        if proc.returncode == 0 and proc.stdout.strip():
            return proc.stdout.strip()
        if proc.returncode != 0:
            return None  # user cancelled Finder dialog

    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        chosen = filedialog.askdirectory(
            initialdir=start,
            title="Select root folder containing PFT Excel files",
        )
        root.destroy()
        if chosen:
            return chosen
    except Exception:
        pass

    print(f"Type folder path and press Enter (default: {start})")
    typed = input("Folder: ").strip()
    return typed or start


def run_gli_batch(
    root: str | Path | None = None,
    *,
    recursive: bool | None = None,
    csv_name: str | None = None,
) -> pd.DataFrame | None:
    """Find all Excel under root, compute GLI, write aggregated CSV."""
    global combined_df, BATCH_ROOT, BATCH_RECURSIVE, BATCH_CSV_NAME

    root_path = Path((root or BATCH_ROOT).strip()).expanduser()
    do_recursive = BATCH_RECURSIVE if recursive is None else recursive
    out_name = csv_name or BATCH_CSV_NAME

    BATCH_ROOT = str(root_path)
    BATCH_RECURSIVE = do_recursive
    BATCH_CSV_NAME = out_name

    if not root_path.is_dir():
        print(f"ERROR: Folder not found:\n  {root_path}")
        return None

    files = discover_excel_files(root_path, recursive=do_recursive)
    if not files:
        print(f"No Excel files under:\n  {root_path}")
        return None

    print(f"Root: {root_path}")
    print(f"Found {len(files)} file(s):")
    for f in files:
        print(f"  • {f.relative_to(root_path)}")

    print("\nRunning GLI (this may take a minute)…")
    combined_df, row_results, skipped = process_excel_paths(
        files,
        mapping=None,
        modules=MODULES,
        auto_mapping_per_file=AUTO_MAPPING_PER_FILE,
    )

    out_path = DEFAULT_OUTPUT_DIR / out_name.strip()
    full_path, summary_path = save_batch_csv_outputs(combined_df, out_path)
    compact_df = build_compact_summary(combined_df)

    print("\n✓ Done")
    print(f"  Rows: {len(combined_df)}  |  Skipped/incomplete: {skipped}")
    if "GLI_PFT_Pattern" in combined_df.columns:
        print("\n  Pattern counts:")
        print(combined_df["GLI_PFT_Pattern"].value_counts().to_string())
    print(f"\n  Full CSV ({len(combined_df.columns)} columns):\n  {full_path}")
    print(f"  Summary CSV ({len(compact_df.columns)} columns):\n  {summary_path}")
    display(compact_df.head(5))
    return combined_df


# Run this cell → folder picker opens, then batch runs
print("Opening folder picker… (Cancel uses BATCH_ROOT from the setup cell)")
_selected = pick_folder()
if _selected:
    print(f"Selected: {_selected}\n")
    run_gli_batch(root=_selected)
else:
    print(f"No folder picked — using BATCH_ROOT:\n  {BATCH_ROOT}\n")
    run_gli_batch()

Opening folder picker… (Cancel uses BATCH_ROOT from the setup cell)
Selected: /Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/data/all/

Root: /Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/data/all
Found 557 file(s):
  • AGT/AGT PFT manual extraction until Mar 2020.xlsx
  • BLAZ/BLAZ PFT 20230426 to 20230609.xlsx
  • BLAZ/BLAZ PFT 20230426 to 20240112.xlsx
  • BLAZ/BLAZ PFT 20230426 to 20240726.xlsx
  • BLAZ/BLAZ PFT 20230426 to 20241025.xlsx
  • BLAZ/BLAZ PFT 20230426 to 20250110.xlsx
  • BLAZ/BLAZ PFT 20230426 to 20250912.xlsx
  • BLAZ/BLAZ PFT 20230426 to 20260228.xlsx
  • BMT/PFT/Archive/BMT PFT 20200720 to 20200828.xlsx
  • BMT/PFT/Archive/BMT PFT 20200831 to 20201002.xlsx
  • BMT/PFT/Archive/BMT PFT 20201005 to 20201030.xlsx
  • BMT/PFT/Archive/BMT PFT 20201102 to 20201106.xlsx
  • BMT/PFT/Archive/BMT PFT 20201109 to 20201113.xlsx
  • BMT/PFT/Archive/BMT PFT 20201116 to 20201120.xlsx
  • BMT/PFT/Archive/BMT PFT 20201123 to 20201127.xlsx
  • BMT/PFT/Archive

/Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/backend/app/services/batch_service.py:551: UserWarning: Discarding nonzero nanoseconds in conversion.
  return pd.to_datetime(val).to_pydatetime()
/Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/backend/app/services/batch_service.py:551: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(val).to_pydatetime()
/Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/backend/app/services/batch_service.py:551: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(val).to_pydatetime()
/Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/backend/app/services/batch_service.py:551: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the defau


✓ Done
  Rows: 13610  |  Skipped/incomplete: 10

  Pattern counts:
GLI_PFT_Pattern
Normal                               7746
Restriction                          3523
Obstruction                          1143
Insufficient data                    1125
Mixed (obstructive + restrictive)      63

  Full CSV (251 columns):
  /Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/data/output/aggregated_PFT_GLI.csv
  Summary CSV (46 columns):
  /Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/data/output/aggregated_PFT_GLI_summary.csv


,source_file,patient_id,patient_id_source,test_date,age,sex,height,FEV1_measured,FEV1_predicted,FEV1_lln,...,RV_status,RVTLC_measured,RVTLC_predicted,RVTLC_lln,RVTLC_z,RVTLC_pct_pred,RVTLC_status,PFT_pattern,PFT_pattern_detail,processing_note
0,AGT PFT manual extraction until Mar 2020.xlsx,None,None,2019-10-04,72.0,F,154.0,NaN,1.847,1.281,...,None,NaN,0.414,0.328,NaN,NaN,None,Insufficient data,Map and provide measured FEV1/FVC and TLC to c...,None
1,AGT PFT manual extraction until Mar 2020.xlsx,None,None,2019-09-19,72.0,F,154.0,NaN,1.847,1.281,...,None,NaN,0.414,0.328,NaN,NaN,None,Insufficient data,Map and provide measured FEV1/FVC and TLC to c...,None
2,AGT PFT manual extraction until Mar 2020.xlsx,None,None,2019-03-29,72.0,F,154.0,NaN,1.847,1.281,...,None,NaN,0.414,0.328,NaN,NaN,None,Insufficient data,Map and provide measured FEV1/FVC and TLC to c...,None
3,AGT PFT manual extraction until Mar 2020.xlsx,None,None,2018-09-13,71.0,F,154.0,NaN,1.866,1.298,...,None,NaN,0.410,0.326,NaN,NaN,None,Insufficient data,Map and provide measured FEV1/FVC and TLC to c...,None
4,AGT PFT manual extraction until Mar 2020.xlsx,None,None,2017-04-05,70.0,F,154.0,NaN,1.885,1.315,...,None,NaN,0.407,0.324,NaN,NaN,None,Insufficient data,Map and provide measured FEV1/FVC and TLC to c...,None


## Preview mapping (first Excel file in folder)

Run after setting the folder above, or set `preview_root` manually.

In [3]:
from backend.app.services.fields import is_manual_pre_export

preview_root = Path(BATCH_ROOT).expanduser()
preview_files = (
    discover_excel_files(preview_root, recursive=BATCH_RECURSIVE)
    if preview_root.is_dir()
    else []
)


def _print_mapping_for(path: Path) -> None:
    cols = [str(c) for c in read_pft_excel(path, nrows=0).columns]
    suggested = suggest_mapping(cols)
    kind = "manual Pre*" if is_manual_pre_export(cols) else "BMT / other"
    print(f"\n{path.name}  [{kind}]")
    for key, col in suggested.model_dump().items():
        if col:
            print(f"  {key:12} → {col}")


if preview_files:
    _print_mapping_for(preview_files[0])
    for path in preview_files:
        cols = [str(c) for c in read_pft_excel(path, nrows=0).columns]
        if is_manual_pre_export(cols) and path != preview_files[0]:
            _print_mapping_for(path)
            break
    print(
        "\nNote: batch uses a separate mapping per file when AUTO_MAPPING_PER_FILE is True."
    )
else:
    print("Set a valid root folder in the interactive cell above.")

First file: BMT PFT 20200720 to 20200828.xlsx (67 columns)

Suggested mapping:
  patient_id   → PATIENT Health Num
  sex          → PATIENT SEX
  dob          → PATIENT DOB
  test_date    → TEST DATE
  height_cm    → TEST HEIGHT
  FEV1         → Spirometry->FEV1;PRE;TESTSELECT;VALUE
  FVC          → Spirometry->FVC;PRE;TESTSELECT;VALUE
  FEV1FVC      → Spirometry->FEV1/FVC;PRE;TESTSELECT;VALUE
  TLC          → Tlc body->TLC;PRE;TESTMEAN;VALUE
  RV           → Tlc body->RV;PRE;TESTMEAN;VALUE
  RVTLC        → Tlc body->RV/TLC;PRE;TESTMEAN;VALUE


## Full results (after interactive run)

Run the interactive cell first. `combined_df` holds all rows.

In [4]:
if combined_df is None or combined_df.empty:
    print("Run the folder picker cell above first.")
else:
    print(f"Rows: {len(combined_df)}  |  Columns: {len(combined_df.columns)}")
    display(combined_df.head(10))

Rows: 9860  |  Columns: 177


,GLI_source_file,GLI_patient_id,GLI_patient_id_source,GLI_test_date,GLI_age_years,GLI_sex,GLI_height_cm,GLI_measured_FEV1,GLI_measured_FVC,GLI_measured_FEV1FVC,...,SVC->IC;PRE;TESTMEAN;VALUE,SVC->IC;PRE;TESTMEAN;%NORM;PC,GLI_IC_Measured,GLI_VC_Measured,PATIENT ID,Spirometry->FEV 3s;NORM,Spirometry->FEV 3s;PRE;TESTMEAN;VALUE,Spirometry->FEV 6s;NORM,Spirometry->FEV 6s;PRE;TESTMEAN;VALUE,Column1
0,BMT PFT 20200720 to 20200828.xlsx,PreBMT-238,PATIENT Health Num,2020-07-20,23.87,M,172.0,2.82,3.53,0.799,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BMT PFT 20200720 to 20200828.xlsx,PreBMT-239,PATIENT Health Num,2020-07-27,55.75,F,165.0,2.72,3.38,0.807,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BMT PFT 20200720 to 20200828.xlsx,PreBMT-240,PATIENT Health Num,2020-07-27,19.99,M,176.5,4.18,4.55,0.919,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BMT PFT 20200720 to 20200828.xlsx,PreBMT-242,PATIENT Health Num,2020-07-29,59.98,F,157.0,1.73,2.08,0.831,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BMT PFT 20200720 to 20200828.xlsx,PreBMT-151,PATIENT Health Num,2020-07-30,62.60,M,175.0,3.54,4.66,0.759,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,BMT PFT 20200720 to 20200828.xlsx,PreBMT-187,PATIENT Health Num,2020-07-30,57.88,F,148.0,2.18,2.46,0.884,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,BMT PFT 20200720 to 20200828.xlsx,PreBMT-243,PATIENT Health Num,2020-08-05,69.59,F,154.0,2.36,2.95,0.799,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,BMT PFT 20200720 to 20200828.xlsx,PreBMT-229,PATIENT Health Num,2020-08-06,66.43,M,199.0,5.03,6.82,0.738,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,BMT PFT 20200720 to 20200828.xlsx,PreBMT-196,PATIENT Health Num,2020-08-06,62.97,M,174.0,2.51,3.20,0.784,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,BMT PFT 20200720 to 20200828.xlsx,PreBMT-003,PATIENT Health Num,2020-08-10,59.06,F,179.0,2.88,3.92,0.733,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Summary: PFT pattern classification

In [5]:
if combined_df is None or combined_df.empty:
    print("Run the interactive batch first.")
elif "GLI_PFT_Pattern" in combined_df.columns:
    display(
        combined_df["GLI_PFT_Pattern"]
        .value_counts(dropna=False)
        .rename("count")
        .to_frame()
    )

    status_cols = [c for c in combined_df.columns if c.endswith("_Status")]
    below_counts = {}
    for col in status_cols:
        param = col.replace("GLI_", "").replace("_Status", "")
        below_counts[param] = (combined_df[col] == "below_lln").sum()
    if below_counts:
        print("\nRows below LLN by parameter:")
        display(pd.Series(below_counts, name="below_lln_count").sort_values(ascending=False))

,count
GLI_PFT_Pattern,
Normal,5879
Restriction,2957
Obstruction,547
Insufficient data,427
Mixed (obstructive + restrictive),44
None,6



Rows below LLN by parameter:


RV         4099
VC         3639
TLC        3001
FVC        3000
FEV1       2639
RVTLC      1930
IC         1651
FEV1FVC     591
FRC           0
ERV           0
Name: below_lln_count, dtype: int64

## Summary preview (same columns as `*_summary.csv`)

In [6]:
if combined_df is None:
    print("No data yet — run the batch cell first.")
else:
    display(build_compact_summary(combined_df).head(20))

,source_file,patient_id,patient_id_source,test_date,age,sex,height,FEV1_measured,FEV1_predicted,FEV1_lln,...,RV_status,RVTLC_measured,RVTLC_predicted,RVTLC_lln,RVTLC_z,RVTLC_pct_pred,RVTLC_status,PFT_pattern,PFT_pattern_detail,processing_note
0,BMT PFT 20200720 to 20200828.xlsx,PreBMT-238,PATIENT Health Num,2020-07-20,23.87,M,172.0,2.82,4.025,3.153,...,normal,0.304,0.298,0.238,0.15,102.1,normal,Restriction,TLC below LLN; FEV1/FVC at or above LLN.,None
1,BMT PFT 20200720 to 20200828.xlsx,PreBMT-239,PATIENT Health Num,2020-07-27,55.75,F,165.0,2.72,2.578,1.874,...,normal,0.335,0.360,0.291,-0.55,93.1,normal,Normal,FEV1/FVC and TLC at or above LLN.,None
2,BMT PFT 20200720 to 20200828.xlsx,PreBMT-240,PATIENT Health Num,2020-07-27,19.99,M,176.5,4.18,4.278,3.362,...,normal,0.286,0.295,0.235,-0.22,97.0,normal,Normal,FEV1/FVC and TLC at or above LLN.,None
3,BMT PFT 20200720 to 20200828.xlsx,PreBMT-242,PATIENT Health Num,2020-07-29,59.98,F,157.0,1.73,2.191,1.573,...,normal,0.476,0.373,0.301,1.86,127.5,above_uln,Normal,FEV1/FVC and TLC at or above LLN.,None
4,BMT PFT 20200720 to 20200828.xlsx,PreBMT-151,PATIENT Health Num,2020-07-30,62.60,M,175.0,3.54,3.174,2.346,...,below_lln,0.231,0.375,0.307,-3.97,61.6,below_lln,Normal,FEV1/FVC and TLC at or above LLN.,None
5,BMT PFT 20200720 to 20200828.xlsx,PreBMT-187,PATIENT Health Num,2020-07-30,57.88,F,148.0,2.18,1.945,1.405,...,normal,0.428,0.367,0.296,1.19,116.7,normal,Normal,FEV1/FVC and TLC at or above LLN.,None
6,BMT PFT 20200720 to 20200828.xlsx,PreBMT-243,PATIENT Health Num,2020-08-05,69.59,F,154.0,2.36,1.893,1.322,...,normal,0.392,0.406,0.323,-0.25,96.6,normal,Normal,FEV1/FVC and TLC at or above LLN.,None
7,BMT PFT 20200720 to 20200828.xlsx,PreBMT-229,PATIENT Health Num,2020-08-06,66.43,M,199.0,5.03,4.190,3.079,...,normal,0.347,0.386,0.314,-0.85,89.8,normal,Normal,FEV1/FVC and TLC at or above LLN.,None
8,BMT PFT 20200720 to 20200828.xlsx,PreBMT-196,PATIENT Health Num,2020-08-06,62.97,M,174.0,2.51,3.117,2.303,...,below_lln,0.343,0.376,0.308,-0.75,91.2,normal,Normal,FEV1/FVC and TLC at or above LLN.,None
9,BMT PFT 20200720 to 20200828.xlsx,PreBMT-003,PATIENT Health Num,2020-08-10,59.06,F,179.0,2.88,3.025,2.178,...,normal,0.334,0.370,0.299,-0.80,90.2,normal,Normal,FEV1/FVC and TLC at or above LLN.,None


## Re-save CSV (optional)

Change filename if needed; uses the same `combined_df` from the interactive run.

In [7]:
if combined_df is None or combined_df.empty:
    print("Nothing to save — run the batch cell first.")
else:
    full_path, summary_path = save_batch_csv_outputs(
        combined_df, DEFAULT_OUTPUT_DIR / BATCH_CSV_NAME.strip()
    )
    print(f"Full:    {full_path}")
    print(f"Summary: {summary_path}")

Full:    /Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/data/output/aggregated_PFT_GLI.csv
Summary: /Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/data/output/aggregated_PFT_GLI_summary.csv
